# Giai đoạn 3: Phân tích Khám phá Dữ liệu & Trực quan hóa (EDA & Visualization)
Dự án: **Phân tích Đối chiếu & Xây dựng Mô hình Cảnh báo Sớm cho Pull Request**
Thành viên thực hiện: **Trần Đức Thịnh (Data Analyst) & Đào Thế Việt (Data Scientist)**

Sổ tay này sử dụng dữ liệu sạch **`data/processed/clean_prs.csv`** để:
- Tính toán các thống kê mô tả chi tiết so sánh giữa FPT University (Local) và Global Projects.
- Trực quan hóa kết quả để trả lời 5 câu hỏi kinh doanh (Business Questions) bằng các biểu đồ chất lượng cao.
- Xuất các biểu đồ kết quả vào thư mục `docs/figures/` để phục vụ báo cáo và làm slide thuyết trình.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Cấu hình đường dẫn tương đối (tự động phát hiện thư mục làm việc)
current_dir = os.getcwd()
if os.path.basename(current_dir) == "notebooks":
    BASE_DIR = os.path.dirname(current_dir)
else:
    BASE_DIR = current_dir

INPUT_PATH = os.path.join(BASE_DIR, "data", "processed", "clean_prs.csv")
FIGURES_DIR = os.path.join(BASE_DIR, "docs", "figures")

# Thiết lập style vẽ biểu đồ sang trọng và chuyên nghiệp
sns.set_theme(style="whitegrid")
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 12

# Đọc dữ liệu sạch
df = pd.read_csv(INPUT_PATH)
df['Project_Type'] = df['is_fpt'].map({1: 'FPT University', 0: 'Global PR'})
print(f"Kích thước dữ liệu sạch: {df.shape}")

### BQ1: Vòng đời Pull Request (PR Lifecycle)
*Câu hỏi:* Thời gian xử lý trung bình từ lúc tạo đến lúc đóng/merge của FPT vs Global có khác biệt không?

In [ ]:
# Tính toán thống kê thời gian sống
bq1_stats = df.groupby('Project_Type')['duration_minutes'].agg(['mean', 'median', 'min', 'max', 'std'])
print("Thống kê thời gian sống (phút):")
print(bq1_stats)

# Trực quan hóa Boxplot thời gian sống (thang đo Log để dễ quan sát phân phối)
plt.figure(figsize=(8, 6))
sns.boxplot(data=df, x='Project_Type', y='duration_minutes', hue='Project_Type', legend=False, palette='pastel')
plt.title('BQ1: So sánh Vòng đời Pull Request (Thang đo Log)', fontsize=14, fontweight='bold', pad=15)
plt.ylabel('Thời gian sống (Phút - Log scale)', fontsize=12)
plt.xlabel('Nhóm dự án', fontsize=12)
plt.yscale('log')

# Lưu biểu đồ
plt.savefig(os.path.join(FIGURES_DIR, "bq1_pr_lifecycle.png"), dpi=300, bbox_inches='tight')
plt.show()

### BQ2: Tỷ lệ Từ chối & Rủi ro (Rejection Rate)
*Câu hỏi:* Tỷ lệ PR bị đóng mà không được merge (Rejected) của FPT vs Global.

In [ ]:
# Tính toán tỷ lệ từ chối
df['is_rejected'] = (df['is_merged'] == 0).astype(int)
bq2_rates = df.groupby('Project_Type')['is_rejected'].mean() * 100
print("Tỷ lệ PR bị từ chối (%):")
print(bq2_rates)

# Vẽ biểu đồ cột so sánh tỷ lệ từ chối
plt.figure(figsize=(7, 5))
ax = sns.barplot(x=bq2_rates.index, y=bq2_rates.values, hue=bq2_rates.index, legend=False, palette='coolwarm')
plt.title('BQ2: Tỷ lệ Pull Request bị từ chối (%)', fontsize=14, fontweight='bold', pad=15)
plt.ylabel('Tỷ lệ từ chối (%)', fontsize=12)
plt.xlabel('Nhóm dự án', fontsize=12)
plt.ylim(0, 12)

# Thêm nhãn số trên mỗi cột
for p in ax.patches:
    ax.annotate(f"{p.get_height():.2f}%", (p.get_x() + p.get_width() / 2., p.get_height() + 0.3),
                ha='center', va='center', fontsize=11, color='black', xytext=(0, 5),
                textcoords='offset points', fontweight='semibold')

plt.savefig(os.path.join(FIGURES_DIR, "bq2_rejection_rates.png"), dpi=300, bbox_inches='tight')
plt.show()

### BQ3: Quy mô Thay đổi Mã nguồn (Code Chunks)
*Câu hỏi:* So sánh quy mô thay đổi (số additions, deletions, changed_files) giữa kỹ sư Global và sinh viên FPT.

In [ ]:
# Tính toán thống kê quy mô
bq3_stats = df.groupby('Project_Type')[['additions', 'deletions', 'changed_files']].mean()
print("Trung bình quy mô thay đổi:")
print(bq3_stats)

# Trực quan hóa phân phối số file thay đổi
plt.figure(figsize=(8, 6))
sns.boxplot(data=df, x='Project_Type', y='changed_files', hue='Project_Type', legend=False, palette='Set3')
plt.title('BQ3: Quy mô sửa đổi (Số lượng file thay đổi)', fontsize=14, fontweight='bold', pad=15)
plt.ylabel('Số lượng file (Log scale)', fontsize=12)
plt.xlabel('Nhóm dự án', fontsize=12)
plt.yscale('log')

plt.savefig(os.path.join(FIGURES_DIR, "bq3_code_scale.png"), dpi=300, bbox_inches='tight')
plt.show()

### BQ4: Văn hóa Kiểm duyệt & Tương tác (Review Culture)
*Câu hỏi:* Mức độ tương tác (số lượng comments và review_comments) trên mỗi PR.

In [ ]:
# Tính thống kê lượng comments
bq4_stats = df.groupby('Project_Type')[['comments', 'review_comments']].mean()
print("Trung bình số lượng bình luận thảo luận & inline review:")
print(bq4_stats)

# Biến đổi định dạng dữ liệu để vẽ biểu đồ nhóm (Grouped Bar Chart)
df_comments = df.melt(id_vars=['Project_Type'], value_vars=['comments', 'review_comments'],
                      var_name='Comment_Type', value_name='Count')
df_comments['Comment_Type'] = df_comments['Comment_Type'].map({
    'comments': 'General Comments',
    'review_comments': 'Inline Review Comments'
})

plt.figure(figsize=(9, 6))
sns.barplot(data=df_comments, x='Project_Type', y='Count', hue='Comment_Type', palette='muted', errorbar=None)
plt.title('BQ4: So sánh Mức độ Tương tác & Review Code', fontsize=14, fontweight='bold', pad=15)
plt.ylabel('Trung bình số bình luận', fontsize=12)
plt.xlabel('Nhóm dự án', fontsize=12)
plt.legend(title="Loại bình luận")

plt.savefig(os.path.join(FIGURES_DIR, "bq4_review_culture.png"), dpi=300, bbox_inches='tight')
plt.show()

### BQ5: Sức ảnh hưởng của các biến (Logistic Regression Weights)
Chúng ta chạy thử nghiệm mô hình Hồi quy Logistic trên tập dữ liệu đã làm sạch để xem trọng số của các thuộc tính tác động đến rủi ro bị reject như thế nào.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Chọn các đặc trưng
features = ['duration_minutes', 'additions', 'deletions', 'changed_files', 'comments', 'review_comments']
X = df[features]
y = df['is_rejected']

# Chuẩn hóa đặc trưng
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Huấn luyện mô hình
model = LogisticRegression(max_iter=1000)
model.fit(X_scaled, y)

# Trích xuất trọng số
importance_df = pd.DataFrame({
    'Feature': features,
    'Weight (Coefficients)': model.coef_[0]
}).sort_values(by='Weight (Coefficients)', key=abs, ascending=False)

print("Trọng số của mô hình Hồi quy Logistic trên dữ liệu sạch:")
print(importance_df)

# Trực quan hóa trọng số
plt.figure(figsize=(8, 6))
sns.barplot(data=importance_df, x='Weight (Coefficients)', y='Feature', hue='Feature', legend=False, palette='coolwarm')
plt.axvline(x=0, color='grey', linestyle='--')
plt.title('BQ5: Trọng số các thuộc tính cảnh báo sớm PR Reject', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Hệ số hồi quy (Trọng số)', fontsize=12)
plt.ylabel('Đặc trưng', fontsize=12)

plt.savefig(os.path.join(FIGURES_DIR, "bq5_feature_importance.png"), dpi=300, bbox_inches='tight')
plt.show()